# EDA: Trader Performance vs Market Sentiment
This notebook analyzes trader performance segmented by the crypto Fear & Greed index.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load datasets
fear_greed_df = pd.read_csv('Datasets/fear_greed_index.csv')
trades_df = pd.read_csv('Datasets/historical_data.csv')


In [2]:
# Preprocess Fear & Greed Index
fear_greed_df['date'] = pd.to_datetime(fear_greed_df['date'])

# Preprocess Trade Data
# Timestamp IST looks like '02-12-2024 22:50'
trades_df['Timestamp IST'] = pd.to_datetime(trades_df['Timestamp IST'], format='mixed', dayfirst=True)
trades_df['date'] = trades_df['Timestamp IST'].dt.normalize()

# Merge Datasets
merged_df = pd.merge(trades_df, fear_greed_df, on='date', how='left')

# Inspect shapes and sample
print(f'Fear Greed shape: {fear_greed_df.shape}')
print(f'Trades shape: {trades_df.shape}')
print(f'Merged shape: {merged_df.shape}')
merged_df.head()


Fear Greed shape: (2644, 4)
Trades shape: (211224, 17)
Merged shape: (211224, 20)


,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,Transaction Hash,Order ID,Crossed,Fee,Trade ID,Timestamp,date,timestamp,value,classification
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,2024-12-02 22:50:00,0.000000,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.345404,8.950000e+14,1.730000e+12,2024-12-02,1.733117e+09,80.0,Extreme Greed
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,2024-12-02 22:50:00,986.524596,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.005600,4.430000e+14,1.730000e+12,2024-12-02,1.733117e+09,80.0,Extreme Greed
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9855,144.09,1150.63,BUY,2024-12-02 22:50:00,1002.518996,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050431,6.600000e+14,1.730000e+12,2024-12-02,1.733117e+09,80.0,Extreme Greed
3,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9874,142.98,1142.04,BUY,2024-12-02 22:50:00,1146.558564,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050043,1.080000e+15,1.730000e+12,2024-12-02,1.733117e+09,80.0,Extreme Greed
4,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9894,8.73,69.75,BUY,2024-12-02 22:50:00,1289.488521,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.003055,1.050000e+15,1.730000e+12,2024-12-02,1.733117e+09,80.0,Extreme Greed


In [3]:
# Clean PnL and Size fields
merged_df['Closed PnL'] = pd.to_numeric(merged_df['Closed PnL'], errors='coerce').fillna(0)
merged_df['Size USD'] = pd.to_numeric(merged_df['Size USD'], errors='coerce').fillna(0)

# UPGRADED: Use pd.cut (vectorized) instead of .apply(map_sentiment) - much faster on 211K rows
merged_df['Sentiment_Group'] = pd.cut(
    merged_df['value'],
    bins=[0, 25, 45, 55, 75, 100],
    labels=['Extreme Fear', 'Fear', 'Neutral', 'Greed', 'Extreme Greed'],
    include_lowest=True
).astype(str)


In [4]:
# Summary Statistics by Sentiment
summary = merged_df.groupby('Sentiment_Group').agg(
    Trade_Count=('Trade ID', 'count'),
    Avg_PnL=('Closed PnL', 'mean'),
    Total_PnL=('Closed PnL', 'sum'),
    Win_Rate=('Closed PnL', lambda x: (x > 0).mean() * 100),
    Avg_Trade_Size=('Size USD', 'mean')
).reset_index()

summary


,Sentiment_Group,Trade_Count,Avg_PnL,Total_PnL,Win_Rate,Avg_Trade_Size
0,Extreme Fear,31364,34.718479,1.088910e+06,34.574672,5132.231840
1,Extreme Greed,32421,74.743267,2.423251e+06,44.369390,3141.881945
2,Fear,55328,57.424206,3.177166e+06,44.563331,8265.791252
3,Greed,55803,41.558959,2.319115e+06,40.334749,5486.158239
4,Neutral,36302,34.324391,1.246044e+06,39.860063,4539.549146
5,nan,6,7078.665688,4.247199e+04,100.000000,14778.143333


## Predictive Modeling (Buy / Sell Recommender)
Transitioning from descriptive analytics to prescriptive analytics. We will build a Deep Neural Network to predict if a trader should go Long or Short given market conditions.

In [5]:
# ML Imports
import joblib
import pickle
import os
import datetime

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input, BatchNormalization
from tensorflow.keras.callbacks import TensorBoard, EarlyStopping

print(f'TensorFlow version: {tf.__version__}')


TensorFlow version: 2.21.0


In [6]:
print('Preparing ML dataset (v2 - richer features)...')
tr_ml = merged_df.copy()

# Standardize Side
tr_ml['Side'] = tr_ml['Side'].str.strip().str.upper()
tr_ml['Side'] = tr_ml['Side'].replace({'BUY': 'Long', 'SELL': 'Short', 'B': 'Long', 'S': 'Short'})
tr_ml = tr_ml[tr_ml['Side'].isin(['Long', 'Short'])]

# Temporal sort
tr_ml = tr_ml.sort_values('Timestamp IST').reset_index(drop=True)

# Ensure numeric
tr_ml['Closed PnL'] = pd.to_numeric(tr_ml['Closed PnL'], errors='coerce').fillna(0)
tr_ml['Size USD'] = pd.to_numeric(tr_ml['Size USD'], errors='coerce').fillna(0)
tr_ml['value'] = pd.to_numeric(tr_ml['value'], errors='coerce').fillna(50)

# ── RICH FEATURE ENGINEERING ─────────────────────────────────────────────
# 1. Temporal features
tr_ml['DayOfWeek'] = tr_ml['Timestamp IST'].dt.dayofweek   # 0=Mon, 6=Sun
tr_ml['HourOfDay'] = tr_ml['Timestamp IST'].dt.hour
tr_ml['IsWeekend'] = (tr_ml['DayOfWeek'] >= 5).astype(int)

# 2. Sentiment bins (numeric rank 0-4)
tr_ml['SentimentRank'] = pd.cut(
    tr_ml['value'],
    bins=[0, 25, 45, 55, 75, 100],
    labels=[0, 1, 2, 3, 4],
    include_lowest=True
).astype(float)

# 3. Log trade size (compresses whale outliers)
tr_ml['LogSizeUSD'] = np.log1p(tr_ml['Size USD'])

# 4. Trade size percentile within each coin
tr_ml['SizePercentile'] = tr_ml.groupby('Coin')['Size USD'].rank(pct=True)

# 5. Rolling win-rate per coin over past 30 trades (look-back, no leakage)
tr_ml['WinFlag'] = (tr_ml['Closed PnL'] > 0).astype(int)
tr_ml['CoinWinRate30'] = (
    tr_ml.groupby('Coin')['WinFlag']
    .transform(lambda x: x.shift(1).rolling(30, min_periods=5).mean())
    .fillna(0.5)  # default = 50% if insufficient history
)

# 6. Interaction: Sentiment x Log Size
tr_ml['SentxSize'] = tr_ml['value'] * tr_ml['LogSizeUSD']

# Filter profitable trades for target
winners = tr_ml[tr_ml['Closed PnL'] > 0].copy()
winners['Target'] = (winners['Side'] == 'Long').astype(int)

# Feature sets
numerical_features = [
    'value', 'LogSizeUSD', 'SentimentRank',
    'DayOfWeek', 'HourOfDay', 'IsWeekend',
    'SizePercentile', 'CoinWinRate30', 'SentxSize'
]
categorical_features = ['Coin']

X = winners[numerical_features + categorical_features]
y = winners['Target']

print(f'Total winning trades: {len(X)}')
print(f'Features: {numerical_features + categorical_features}')
print(f'Target Distribution (Long=1, Short=0):')
print(y.value_counts(normalize=True))


Preparing ML dataset (v2 - richer features)...
Total winning trades: 86869
Features: ['value', 'LogSizeUSD', 'SentimentRank', 'DayOfWeek', 'HourOfDay', 'IsWeekend', 'SizePercentile', 'CoinWinRate30', 'SentxSize', 'Coin']
Target Distribution (Long=1, Short=0):
Target
0    0.676191
1    0.323809
Name: proportion, dtype: float64


In [7]:
# Temporal split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=False
)

print('Building Preprocessor...')
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f'Train size: {len(X_train)} | Test size: {len(X_test)}')
print(f'Input feature dim: {X_train_processed.shape[1]}')

os.makedirs('models', exist_ok=True)
with open('models/preprocessor.pkl', 'wb') as f:
    pickle.dump(preprocessor, f)

# Also save the feature list for use in app.py
import json as _json
meta = {
    'numerical_features': numerical_features,
    'categorical_features': categorical_features
}
with open('models/feature_meta.json', 'w') as f:
    _json.dump(meta, f)
print('Saved preprocessor and feature_meta.json')


Building Preprocessor...
Train size: 69495 | Test size: 17374
Input feature dim: 196
Saved preprocessor and feature_meta.json


In [8]:
# ─── BASELINE: Random Forest ──────────────────────────────────────────
print("\nTraining Baseline Random Forest...")
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train_processed, y_train)

rf_preds = rf.predict(X_test_processed)
print("Random Forest Accuracy:", accuracy_score(y_test, rf_preds))
print(classification_report(y_test, rf_preds, target_names=["Short", "Long"]))

# Save RF model
with open('models/rf_baseline.pkl', 'wb') as f:
    pickle.dump(rf, f)
print("Saved Baseline Random Forest model to models/rf_baseline.pkl")


Training Baseline Random Forest...
Random Forest Accuracy: 0.5905951421664556
              precision    recall  f1-score   support

       Short       0.56      0.99      0.72      9130
        Long       0.92      0.15      0.26      8244

    accuracy                           0.59     17374
   macro avg       0.74      0.57      0.49     17374
weighted avg       0.73      0.59      0.50     17374

Saved Baseline Random Forest model to models/rf_baseline.pkl


In [9]:
# NEURAL NETWORK (FNN) v2 - Richer features, no class_weight collapse
import datetime
from tensorflow.keras.callbacks import TensorBoard
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

print('Training Neural Network v2...')
input_dim = X_train_processed.shape[1]
print(f'Input dim: {input_dim}')

model = Sequential([
    Input(shape=(input_dim,)),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.25),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# TensorBoard
log_dir = os.path.join('logs', 'fit', datetime.datetime.now().strftime('%Y%m%d-%H%M%S'))
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

# NOTE: No class_weight here — it was pushing the model to predict uniform output
history = model.fit(
    X_train_processed, y_train,
    validation_split=0.2,
    epochs=500,
    batch_size=512,
    callbacks=[tensorboard_callback],
    verbose=1
)

nn_loss, nn_acc, nn_auc = model.evaluate(X_test_processed, y_test, verbose=0)
print(f'Neural Network => Accuracy: {nn_acc:.4f} | AUC: {nn_auc:.4f}')

model.save('models/trade_predictor_nn.h5')
print('Saved to models/trade_predictor_nn.h5')


Training Neural Network v2...
Input dim: 196
Epoch 1/500
109/109 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.7575 - auc: 0.8133 - loss: 0.4641 - val_accuracy: 0.6517 - val_auc: 0.7119 - val_loss: 0.6600
Epoch 2/500
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8327 - auc: 0.9078 - loss: 0.3448 - val_accuracy: 0.3289 - val_auc: 0.6355 - val_loss: 0.7510
Epoch 3/500
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8595 - auc: 0.9315 - loss: 0.2997 - val_accuracy: 0.4714 - val_auc: 0.6444 - val_loss: 0.7360
Epoch 4/500
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8715 - auc: 0.9431 - loss: 0.2745 - val_accuracy: 0.5159 - val_auc: 0.6468 - val_loss: 0.7672
Epoch 5/500
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8826 - auc: 0.9514 - loss: 0.2547 - val_accuracy: 0.6067 - val_auc: 0.6642 - val_loss: 0.7521
Epoch 6/500
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8877 - auc: 0.9558 - loss: 0.2434 - val_accuracy: 0.5580 - val_auc: 0.6403 - val_l

Neural Network => Accuracy: 0.5707 | AUC: 0.6220
Saved to models/trade_predictor_nn.h5


In [10]:
# MODEL COMPARISON SUMMARY
print('='*60)
print('  MODEL COMPARISON SUMMARY')
print('='*60)
print(f'  Random Forest:    Accuracy = {accuracy_score(y_test, rf_preds):.4f}')
print(f'  Neural Network:   Accuracy = {nn_acc:.4f}')
print('='*60)
print('  Saved -> models/trade_predictor_nn.h5')
print('  Saved -> models/preprocessor.pkl')
print('  Saved -> models/rf_baseline.pkl')


  MODEL COMPARISON SUMMARY
  Random Forest:    Accuracy = 0.5906
  Neural Network:   Accuracy = 0.5707
  Saved -> models/trade_predictor_nn.h5
  Saved -> models/preprocessor.pkl
  Saved -> models/rf_baseline.pkl
